# Kitchen zone AI - training

1. **Runtime > Change runtime type > T4 GPU**, then **Save**.
2. **Runtime > Run all**.
3. At the end the browser downloads `kitchen_model.zip` (~10 MB).
   Unzip it into `Interior/backend/models/` on your PC.

Steps: download free-licence kitchen photos (Openverse) > auto-label with
SegFormer ADE20K > train YOLO11n-seg > export ONNX.
Takes about 30-60 minutes on the free T4.

In [ ]:
MAX_IMAGES = 600   # photos to download
EPOCHS = 80
UPLOAD_OWN_PHOTOS = False  # True = pick your own kitchen photos to add

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics

In [ ]:
%%writefile kitchen_classes.py
"""Kitchen zone classes shared by auto-labelling, training and the backend."""

KITCHEN_CLASSES = [
    "wall",
    "floor",
    "ceiling",
    "cabinet",
    "countertop",
    "stove",
    "sink",
    "fridge",
    "window",
    "hood",
    "door",
    # Open space under a bare platform where base cabinets go. Hand-labelled
    # only: ADE20K has no such class.
    "under_counter",
]

# ADE20K label name (as in the HF SegFormer config) -> kitchen class.
ADE_TO_KITCHEN = {
    "wall": "wall",
    "floor": "floor",
    "rug": "floor",
    "ceiling": "ceiling",
    "cabinet": "cabinet",
    "wardrobe": "cabinet",
    "chest of drawers": "cabinet",
    "shelf": "cabinet",
    "countertop": "countertop",
    "counter": "countertop",
    "kitchen island": "countertop",
    "stove": "stove",
    "oven": "stove",
    "sink": "sink",
    "refrigerator": "fridge",
    "windowpane": "window",
    "hood": "hood",
    "door": "door",
    "screen door": "door",
}

# Classes that prove the photo is a kitchen (used to reject living rooms etc.).
KITCHEN_EVIDENCE = {"cabinet", "countertop", "stove", "sink", "hood", "fridge"}

# Paint order for zone maps: big surfaces first, small objects on top.
PAINT_ORDER = [
    "ceiling",
    "wall",
    "floor",
    "door",
    "window",
    "under_counter",
    "cabinet",
    "fridge",
    "countertop",
    "hood",
    "stove",
    "sink",
]

In [ ]:
%%writefile download.py
"""Download commercially usable kitchen photos from Openverse (no API key).

Writes resized JPEGs plus attribution.csv (required by CC-BY / CC-BY-SA).
"""

import argparse
import csv
import io
import time
from pathlib import Path

import numpy as np
import requests
from PIL import Image

API = "https://api.openverse.org/v1/images/"
HEADERS = {"User-Agent": "InteriorKitchenAI/0.1 (kitchen layout dataset)"}
QUERIES = [
    "kitchen",
    "modular kitchen",
    "kitchen interior",
    "kitchen cabinets",
    "kitchen design",
    "indian kitchen",
    "kitchen counter",
    "small kitchen",
    "empty kitchen",
    "kitchen renovation",
    "open kitchen",
    "kitchen chimney",
]
MAX_PAGES = 12
MAX_SIDE = 1024
MIN_SIDE = 400


def dhash(img: Image.Image) -> int:
    g = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    bits = 0
    for b in (g[:, 1:] > g[:, :-1]).flatten():
        bits = (bits << 1) | int(b)
    return bits


def is_duplicate(h: int, seen: list[int], max_dist: int = 6) -> bool:
    return any(bin(h ^ s).count("1") <= max_dist for s in seen)


def save_resized(img: Image.Image, path: Path) -> None:
    img = img.convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
    img.save(path, "JPEG", quality=90)


def fetch_image(session: requests.Session, item: dict) -> Image.Image | None:
    for url in (item.get("url"), item.get("thumbnail")):
        if not url:
            continue
        try:
            r = session.get(url, timeout=30)
            if r.status_code != 200:
                continue
            img = Image.open(io.BytesIO(r.content))
            img.load()
            return img
        except Exception:
            continue
    return None


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/raw")
    ap.add_argument("--max", type=int, default=600)
    ap.add_argument("--extra", default=None, help="folder of your own photos to include")
    args = ap.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    session.headers.update(HEADERS)

    seen_ids: set[str] = set()
    seen_hashes: list[int] = []
    rows: list[dict] = []

    if args.extra and Path(args.extra).is_dir():
        for p in sorted(Path(args.extra).iterdir()):
            if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
                continue
            img = Image.open(p)
            seen_hashes.append(dhash(img))
            dest = out / f"own_{p.stem}.jpg"
            save_resized(img, dest)
            rows.append({"file": dest.name, "source": "own", "license": "own"})
        print(f"included {len(rows)} own photos from {args.extra}")

    for q in QUERIES:
        if len(rows) >= args.max:
            break
        for page in range(1, MAX_PAGES + 1):
            if len(rows) >= args.max:
                break
            params = {
                "q": q,
                "license_type": "commercial",
                "category": "photograph",
                "mature": "false",
                "page_size": 20,
                "page": page,
            }
            try:
                r = session.get(API, params=params, timeout=30)
            except requests.RequestException as e:
                print(f"  api error {e}; skipping")
                break
            if r.status_code != 200:
                print(f"  '{q}' page {page}: HTTP {r.status_code}, next query")
                break
            results = r.json().get("results", [])
            if not results:
                break
            for item in results:
                if len(rows) >= args.max:
                    break
                if item["id"] in seen_ids:
                    continue
                seen_ids.add(item["id"])
                if min(item.get("width") or 0, item.get("height") or 0) < MIN_SIDE and item.get("width"):
                    continue
                img = fetch_image(session, item)
                if img is None or min(img.size) < 256:
                    continue
                h = dhash(img)
                if is_duplicate(h, seen_hashes):
                    continue
                seen_hashes.append(h)
                name = f"ov_{item['id'][:12]}.jpg"
                save_resized(img, out / name)
                rows.append(
                    {
                        "file": name,
                        "source": item.get("source", ""),
                        "license": f"{item.get('license', '')} {item.get('license_version', '')}".strip(),
                        "license_url": item.get("license_url", ""),
                        "title": item.get("title", ""),
                        "creator": item.get("creator", ""),
                        "landing_url": item.get("foreign_landing_url", ""),
                    }
                )
                time.sleep(0.15)
            print(f"'{q}' page {page}: total {len(rows)}")
            time.sleep(1.0)

    fields = ["file", "source", "license", "license_url", "title", "creator", "landing_url"]
    with open(out / "attribution.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for row in rows:
            w.writerow({k: row.get(k, "") for k in fields})
    print(f"done: {len(rows)} images in {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile autolabel.py
"""Auto-label kitchen photos with a pretrained ADE20K SegFormer and write a
YOLO-seg dataset (polygons per kitchen class) plus preview overlays."""

import argparse
import hashlib
import json
import shutil
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation

from kitchen_classes import ADE_TO_KITCHEN, KITCHEN_CLASSES, KITCHEN_EVIDENCE

IGNORE = 255
PALETTE = np.array(
    [
        [200, 200, 200],
        [120, 80, 40],
        [240, 240, 180],
        [30, 144, 255],
        [255, 140, 0],
        [220, 20, 60],
        [0, 206, 209],
        [148, 0, 211],
        [173, 255, 47],
        [255, 20, 147],
        [139, 69, 19],
        [255, 215, 0],
    ],
    dtype=np.uint8,
)


def build_lut(id2label: dict) -> np.ndarray:
    lut = np.full(256, IGNORE, dtype=np.uint8)
    found = set()
    for ade_id, name in id2label.items():
        kc = ADE_TO_KITCHEN.get(name.strip().lower())
        if kc is not None:
            lut[int(ade_id)] = KITCHEN_CLASSES.index(kc)
            found.add(name.strip().lower())
    missing = set(ADE_TO_KITCHEN) - found
    if missing:
        print(f"warning: ADE labels not in model config: {sorted(missing)}")
    return lut


def mask_to_polygons(mask: np.ndarray, min_area: float) -> list[np.ndarray]:
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for c in contours:
        if cv2.contourArea(c) < min_area:
            continue
        eps = 0.0015 * cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, eps, True).reshape(-1, 2)
        if len(approx) >= 3:
            polys.append(approx)
    return polys


def split_of(name: str, val_ratio: float) -> str:
    h = int(hashlib.md5(name.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "val" if h < val_ratio else "train"


def add_manual(out: Path, folder: str | None, repeat: int) -> int:
    """Copy LabelMe-labelled photos into train, repeated so they outweigh
    the noisier auto-labels."""
    if not folder or not Path(folder).is_dir():
        return 0
    count = 0
    for js in sorted(Path(folder).glob("*.json")):
        data = json.loads(js.read_text(encoding="utf-8"))
        img = Image.open(js.parent / data["imagePath"]).convert("RGB")
        w, h = img.size
        lines = []
        for shape in data["shapes"]:
            if shape["label"] not in KITCHEN_CLASSES:
                print(f"  {js.name}: unknown label '{shape['label']}' skipped")
                continue
            pts = shape["points"]
            if shape.get("shape_type") == "rectangle":
                (x0, y0), (x1, y1) = pts
                pts = [[x0, y0], [x1, y0], [x1, y1], [x0, y1]]
            coords = " ".join(
                f"{min(max(x / w, 0), 1):.5f} {min(max(y / h, 0), 1):.5f}" for x, y in pts
            )
            lines.append(f"{KITCHEN_CLASSES.index(shape['label'])} {coords}")
        for i in range(repeat):
            stem = f"manual_{js.stem}_{i}"
            img.save(out / "images" / "train" / f"{stem}.jpg", "JPEG", quality=92)
            (out / "labels" / "train" / f"{stem}.txt").write_text("\n".join(lines))
        count += 1
    return count


@torch.inference_mode()
def segment(model, processor, img: Image.Image, device: str) -> np.ndarray:
    inputs = processor(images=img, return_tensors="pt").to(device)
    logits = model(**inputs).logits
    up = torch.nn.functional.interpolate(
        logits, size=(img.height, img.width), mode="bilinear", align_corners=False
    )
    return up.argmax(1)[0].to("cpu").numpy().astype(np.uint8)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--images", default="data/raw")
    ap.add_argument("--out", default="data/dataset")
    ap.add_argument("--model", default="nvidia/segformer-b5-finetuned-ade-640-640")
    ap.add_argument("--min-evidence", type=float, default=0.05)
    ap.add_argument("--min-shell", type=float, default=0.55)
    ap.add_argument("--val-ratio", type=float, default=0.15)
    ap.add_argument("--previews", type=int, default=40)
    ap.add_argument("--manual", default=None, help="folder of LabelMe JSON + photos")
    ap.add_argument("--manual-repeat", type=int, default=15)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"loading {args.model} on {device}")
    processor = AutoImageProcessor.from_pretrained(args.model)
    model = SegformerForSemanticSegmentation.from_pretrained(args.model).to(device).eval()
    lut = build_lut(model.config.id2label)
    evidence_ids = [KITCHEN_CLASSES.index(c) for c in KITCHEN_EVIDENCE]
    shell_ids = [KITCHEN_CLASSES.index(c) for c in ("wall", "floor")]

    out = Path(args.out)
    if out.exists():
        shutil.rmtree(out)
    for split in ("train", "val"):
        (out / "images" / split).mkdir(parents=True)
        (out / "labels" / split).mkdir(parents=True)
    preview_dir = out / "previews"
    preview_dir.mkdir()

    files = sorted(
        p for p in Path(args.images).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    kept, rejected, previews = 0, [], 0
    for i, path in enumerate(files, 1):
        img = Image.open(path).convert("RGB")
        kmap = lut[segment(model, processor, img, device)]
        evidence = np.isin(kmap, evidence_ids).mean()
        # Empty "before" rooms have no cabinets yet but a clear wall + floor shell.
        shell = np.isin(kmap, shell_ids).mean()
        if evidence < args.min_evidence and shell < args.min_shell:
            rejected.append(path.name)
            continue

        h, w = kmap.shape
        min_area = 0.004 * h * w
        lines = []
        for cls_id in range(len(KITCHEN_CLASSES)):
            mask = (kmap == cls_id).astype(np.uint8)
            if mask.sum() < min_area:
                continue
            for poly in mask_to_polygons(mask, min_area):
                coords = " ".join(f"{x / w:.5f} {y / h:.5f}" for x, y in poly)
                lines.append(f"{cls_id} {coords}")
        if not lines:
            rejected.append(path.name)
            continue

        split = split_of(path.name, args.val_ratio)
        img.save(out / "images" / split / f"{path.stem}.jpg", "JPEG", quality=92)
        (out / "labels" / split / f"{path.stem}.txt").write_text("\n".join(lines))
        kept += 1

        if previews < args.previews:
            color = np.zeros((h, w, 3), np.uint8)
            valid = kmap != IGNORE
            color[valid] = PALETTE[kmap[valid]]
            blend = (np.asarray(img) * 0.5 + color * 0.5).astype(np.uint8)
            Image.fromarray(blend).save(preview_dir / f"{path.stem}.jpg", quality=85)
            previews += 1
        if i % 20 == 0:
            print(f"{i}/{len(files)} processed, kept {kept}")

    manual = add_manual(out, args.manual, args.manual_repeat)
    print(f"hand-labelled photos: {manual} (x{args.manual_repeat} in train)")

    val_imgs = list((out / "images" / "val").iterdir())
    train_imgs = sorted((out / "images" / "train").iterdir())
    if not val_imgs and len(train_imgs) > 1:
        moved = train_imgs[0]
        shutil.move(moved, out / "images" / "val" / moved.name)
        label = out / "labels" / "train" / f"{moved.stem}.txt"
        shutil.move(label, out / "labels" / "val" / label.name)

    (out / "data.yaml").write_text(
        f"path: {out.resolve().as_posix()}\n"
        "train: images/train\n"
        "val: images/val\n"
        f"names: {json.dumps(KITCHEN_CLASSES)}\n"
    )
    (out / "rejected.txt").write_text("\n".join(rejected))
    print(f"done: kept {kept}, rejected {len(rejected)} (not kitchen / empty)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""Train the kitchen zone segmenter (YOLO-seg) and export it to ONNX."""

import argparse
import json
import shutil
from pathlib import Path

from ultralytics import YOLO

from kitchen_classes import KITCHEN_CLASSES, PAINT_ORDER


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/dataset/data.yaml")
    ap.add_argument("--base", default="yolo11n-seg.pt")
    ap.add_argument("--epochs", type=int, default=80)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--device", default=None)
    ap.add_argument("--out", default="export")
    args = ap.parse_args()

    model = YOLO(args.base)
    model.train(
        data=args.data,
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        device=args.device,
        project="runs",
        name="kitchen_seg",
        exist_ok=True,
        patience=25,
        fliplr=0.5,
        degrees=3.0,
        perspective=0.0005,
        plots=True,
    )

    best = Path(model.trainer.save_dir) / "weights" / "best.pt"
    trained = YOLO(str(best))
    metrics = trained.val(data=args.data, imgsz=args.imgsz, device=args.device)
    onnx_path = Path(trained.export(format="onnx", imgsz=args.imgsz, opset=12, simplify=True))

    out = Path(args.out)
    out.mkdir(exist_ok=True)
    shutil.copy(best, out / "kitchen_seg.pt")
    shutil.copy(onnx_path, out / "kitchen_seg.onnx")
    (out / "kitchen_seg.json").write_text(
        json.dumps(
            {
                "classes": KITCHEN_CLASSES,
                "paintOrder": PAINT_ORDER,
                "imgsz": args.imgsz,
                "maskMap50_95": float(metrics.seg.map),
                "boxMap50_95": float(metrics.box.map),
            },
            indent=2,
        )
    )
    print(f"exported to {out.resolve()}  mask mAP50-95={metrics.seg.map:.3f}")


if __name__ == "__main__":
    main()

Hand-labelled site photos embedded from `datasets/labelled` (1 photos).

In [ ]:
import base64, os
os.makedirs('labelled', exist_ok=True)
LABELLED = {"site_empty_L_01.jpg": "...", "site_empty_L_01.json": "ew0KICAidmVyc2lvbiI6ICI1LjQuMSIsDQogICJmbGFncyI6IHt9LA0KICAiaW1hZ2VQYXRoIjogInNpdGVfZW1wdHlfTF8wMS5qcGciLA0KICAiaW1hZ2VEYXRhIjogbnVsbCwNCiAgImltYWdlV2lkdGgiOiAxMDI0LA0KICAiaW1hZ2VIZWlnaHQiOiA2ODIsDQogICJzaGFwZXMiOiBbDQogICAgeyAibGFiZWwiOiAiY2VpbGluZyIsICJzaGFwZV90eXBlIjogInBvbHlnb24iLCAicG9pbnRzIjogW1syMDUsIDBdLCBbODIyLCAwXSwgWzcxNiwgNzVdLCBbMjQ0LCA3Ml1dIH0sDQogICAgeyAibGFiZWwiOiAid2FsbCIsICJzaGFwZV90eXBlIjogInBvbHlnb24iLCAicG9pbnRzIjogW1syNDQsIDcyXSwgWzcxNiwgNzVdLCBbNzE0LCAzNDddLCBbMjI1LCAzNDhdXSB9LA0KICAgIHsgImxhYmVsIjogIndhbGwiLCAic2hhcGVfdHlwZSI6ICJwb2x5Z29uIiwgInBvaW50cyI6IFtbODAsIDBdLCBbMjA1LCAwXSwgWzI0NCwgNzJdLCBbMjI1LCAzNDhdLCBbMjIyLCA0OTBdLCBbMTE4LCA2MjVdLCBbODAsIDYyNV1dIH0sDQogICAgeyAibGFiZWwiOiAid2FsbCIsICJzaGFwZV90eXBlIjogInBvbHlnb24iLCAicG9pbnRzIjogW1s3MTYsIDc1XSwgWzgyMiwgMF0sIFsxMDI0LCAwXSwgWzEwMjQsIDQzMF0sIFs3MTUsIDM0OF1dIH0sDQogICAgeyAibGFiZWwiOiAiZG9vciIsICJzaGFwZV90eXBlIjogInBvbHlnb24iLCAicG9pbnRzIjogW1swLCAxMl0sIFs4MCwgNjhdLCBbODAsIDYyNV0sIFswLCA2MzJdXSB9LA0KICAgIHsgImxhYmVsIjogIndpbmRvdyIsICJzaGFwZV90eXBlIjogInBvbHlnb24iLCAicG9pbnRzIjogW1szNjMsIDE3OV0sIFs1NTAsIDE3OV0sIFs1NTAsIDMwNV0sIFszNjMsIDMwNV1dIH0sDQogICAgeyAibGFiZWwiOiAiY291bnRlcnRvcCIsICJzaGFwZV90eXBlIjogInBvbHlnb24iLCAicG9pbnRzIjogW1syMjUsIDM0N10sIFs3MTUsIDM0N10sIFsxMDE1LCA0MjhdLCBbMTAxNSwgNDU3XSwgWzgzOCwgNDU4XSwgWzgzOCwgNDQ2XSwgWzY1OCwgMzc0XSwgWzIyNSwgMzc0XV0gfSwNCiAgICB7ICJsYWJlbCI6ICJzaW5rIiwgInNoYXBlX3R5cGUiOiAicG9seWdvbiIsICJwb2ludHMiOiBbWzMwMiwgMzQ4XSwgWzM3OCwgMzQ4XSwgWzM3NiwgMzU3XSwgWzMwNiwgMzU3XV0gfSwNCiAgICB7ICJsYWJlbCI6ICJ1bmRlcl9jb3VudGVyIiwgInNoYXBlX3R5cGUiOiAicG9seWdvbiIsICJwb2ludHMiOiBbWzIyNSwgMzc1XSwgWzY1MiwgMzc1XSwgWzY1MiwgNDg4XSwgWzIyNSwgNDkwXV0gfSwNCiAgICB7ICJsYWJlbCI6ICJ1bmRlcl9jb3VudGVyIiwgInNoYXBlX3R5cGUiOiAicG9seWdvbiIsICJwb2ludHMiOiBbWzY3MiwgMzgyXSwgWzgzOCwgNDYwXSwgWzgzOCwgNjY2XSwgWzgwNiwgNjY2XSwgWzY3MiwgNDkyXV0gfSwNCiAgICB7ICJsYWJlbCI6ICJmbG9vciIsICJzaGFwZV90eXBlIjogInBvbHlnb24iLCAicG9pbnRzIjogW1s4MCwgNjI1XSwgWzExOCwgNjI1XSwgWzIyMiwgNDkyXSwgWzY2OCwgNDk0XSwgWzgwNiwgNjY4XSwgWzEwMjQsIDY2OF0sIFsxMDI0LCA2ODJdLCBbMCwgNjgyXSwgWzAsIDYzMl1dIH0NCiAgXQ0KfQ0K"}
for name, data in LABELLED.items():
    open(os.path.join('labelled', name), 'wb').write(base64.b64decode(data))
print(len(LABELLED), 'files')

In [ ]:
import os
os.makedirs('own', exist_ok=True)
if UPLOAD_OWN_PHOTOS:
    from google.colab import files
    for name, data in files.upload().items():
        open(os.path.join('own', name), 'wb').write(data)
print(len(os.listdir('own')), 'own photos')

In [ ]:
!python download.py --out data/raw --max {MAX_IMAGES} --extra own

In [ ]:
!python autolabel.py --images data/raw --out data/dataset --manual labelled --manual-repeat 20

Check the auto-labels look sensible (colours = wall / floor / cabinet / countertop ...).

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('data/dataset/previews/*.jpg'))[:8]:
    display(Image(filename=p, width=480))

In [ ]:
!python train.py --data data/dataset/data.yaml --epochs {EPOCHS} --batch 16

In [ ]:
import shutil
shutil.copy('data/raw/attribution.csv', 'export/attribution.csv')
shutil.make_archive('kitchen_model', 'zip', 'export')
from google.colab import files
files.download('kitchen_model.zip')